# Geopack SDK: ESRI Geodatabase Manager

Python equivalent of the portal **ESRI Geodatabase Manager** dialog (`EsriGeodatabaseDataStoreManager.vue`) for a datastore such as **GDB_Golbahar** (ID 31).

| Portal action | SDK call |
|---------------|----------|
| Refresh Discovery | `discover_esri_datasets(id)` |
| Register (one) | `register_esri_datasets(id, wg_id, dataset_names=[name])` |
| Register All | `register_esri_datasets(id, wg_id)` |
| Update All Schemas | `update_esri_dataset_schemas(id)` |
| Delete All Datasets | `delete_all_esri_datasets(id, confirm=True)` |
| Status / counts | `get_esri_geodatabase_info(id)` |

`.env`: `GEOPACK_*`, `TEST_ESRI_DATASTORE_ID`, `TEST_WORKGROUP_ID`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from dotenv import load_dotenv

source_path = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if os.path.exists(source_path) and source_path not in sys.path:
    sys.path.insert(0, source_path)

from geopack_sdk import GeopackClient
from geopack_sdk.esri_geodatabase import (
    filter_datasets,
    group_datasets_by_feature_dataset,
    print_discovery_table,
    print_registration_summary,
    print_schema_update_summary,
)

load_dotenv()
client = GeopackClient(base_url=os.getenv("GEOPACK_API_URL", "http://localhost:3000/api"))
client.auth.login(
    os.getenv("GEOPACK_USERNAME", "admin"),
    os.getenv("GEOPACK_PASSWORD", "password"),
)
print("Logged in as", client.users.me().userName)

## 1. Select ESRI datastore (like opening the manager for ID 31)

In [ ]:
ESRI_DATASTORE_ID = int(os.getenv("TEST_ESRI_DATASTORE_ID", "31"))
WORKGROUP_ID = int(os.getenv("TEST_WORKGROUP_ID", "1"))

store = client.datastores.get(ESRI_DATASTORE_ID)
print(f"Datastore #{store.id}: {store.name} ({store.type}) — {store.status}")

## 2. Geodatabase info

In [ ]:
info = client.datastores.get_esri_geodatabase_info(ESRI_DATASTORE_ID)
gdb = info.data.geodatabase
print("Version:", gdb.version)
print("Feature classes:", gdb.featureClassCount, "| Tables:", gdb.tableCount)

## 3. Refresh discovery

In [ ]:
discovery = client.datastores.discover_esri_datasets(ESRI_DATASTORE_ID)
print_discovery_table(discovery, max_rows_per_category=8)

## 4. Search & browse by feature dataset

In [ ]:
SEARCH = ""  # e.g. "BTS" or "Communication"
filtered = filter_datasets(discovery.data, SEARCH)
for cat in group_datasets_by_feature_dataset(filtered)[:5]:
    print(f"{cat.name} ({len(cat.datasets)}) — {cat.category_type}")

## 5. Register one dataset (portal **Register** button)

Uncomment and set a name from the discovery table (`dataset.name`).

In [ ]:
# SINGLE_NAME = "GDB_Golbahar.DBO.BTS"
# reg = client.datastores.register_esri_datasets(
#     ESRI_DATASTORE_ID,
#     workgroup_id=WORKGROUP_ID,
#     dataset_names=[SINGLE_NAME],
# )
# print_registration_summary(reg)
# discovery = client.datastores.discover_esri_datasets(ESRI_DATASTORE_ID)

## 6. Register all — use with care

In [ ]:
# RUN_REGISTER_ALL = False
# if RUN_REGISTER_ALL:
#     reg_all = client.datastores.register_esri_datasets(
#         ESRI_DATASTORE_ID, workgroup_id=WORKGROUP_ID
#     )
#     print_registration_summary(reg_all)

## 7. Update schemas / delete all (destructive)

In [ ]:
# schema = client.datastores.update_esri_dataset_schemas(
#     ESRI_DATASTORE_ID, force_update=True
# )
# print_schema_update_summary(schema)

# deleted = client.datastores.delete_all_esri_datasets(
#     ESRI_DATASTORE_ID, confirm=True
# )
# print(len(deleted.data.deleted), "deleted")

## 8. CLI test

`python test_esri_datastore.py`